In [1]:
import warnings
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.pipeline import make_pipeline
import moabb
from moabb.datasets import BNCI2014_001
from moabb.evaluations import WithinSessionEvaluation as WSE
from moabb.paradigms import LeftRightImagery

#We need to set up the instantiation process of the dataset
moabb.set_log_level("info")
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'mne'

In [2]:
import mne

ModuleNotFoundError: No module named 'mne'

In [3]:
import scikit

ModuleNotFoundError: No module named 'scikit'

In [4]:
pip install mne

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 6.0 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 4.9 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 8.2 MB/s eta 0:00:00m eta 0:00:010:00:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.7/22.7 MB 8.7 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.3
    Uninstalling numpy-1.24.3:
      Successfully uninstalled numpy-1.24.3
  Attempting uninstall: scipy
    Found existing installation: scipy 1.10.1
    Uninstalling scipy-1.10.1:
      Successfully uninstalled scipy-1.10.1
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.7.1
    Uninstalling matplotlib-3.7.1:
      Successfully uninstalled matplotlib-3.7.1
Note: you may need to restart the kernel to use updated

In [ ]:
import mne

raw = mne.io.read_raw_csv('/content/sample.edf', preload=True)
raw.plot(n_channels=10, scalings='auto')

In [ ]:
bands = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 12),
    'beta': (12, 30),
}

In [ ]:
import numpy as np

band_powers = {}

for band_name, (low, high) in bands.items():
    power = raw.copy().filter(low, high).get_data().var(axis=1).mean()
    band_powers[band_name] = power

band_powers

In [ ]:
import pandas as pd

df_row = pd.DataFrame([band_powers])
df_row["label"] = 0   # 0 = normal (left), 1 = abnormal(right)
df_row

In [ ]:
def extract_band_powers(raw):
    bands = {
        'delta': (0.5, 4),
        'theta': (4, 8),
        'alpha': (8, 12),
        'beta': (12, 30),
    }
    
    powers = {}
    for band, (low, high) in bands.items():
        filtered = raw.copy().filter(low, high)
        power = filtered.get_data().var(axis=1).mean()
        powers[band] = power
    return powers

In [ ]:
data_rows = []

base_path = "/content/eeg_data/TKS - Building An EEG-Based Classifier for Brain Abnormality Detection"

for label_folder, label_value in [("normal", 0), ("abnormal", 1)]:
    folder_path = os.path.join(base_path, label_folder)

    for filename in os.listdir(folder_path):
        if filename.endswith(".edf"):
            filepath = os.path.join(folder_path, filename)
            raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)
            band_powers = extract_band_powers(raw)
            band_powers["label"] = label_value
            data_rows.append(band_powers)

df = pd.DataFrame(data_rows)
df

In [ ]:
from sklearn.model_selection import train_test_split

X = df[['delta', 'theta', 'alpha', 'beta']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# train baseline models 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

log_reg = LogisticRegression()
rf = RandomForestClassifier(n_estimators=100, random_state=42)

log_reg.fit(X_train, y_train)
rf.fit(X_train, y_train)

In [ ]:
# evaluate each model using accuracy and a confusion matrix 
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred_log = log_reg.predict(X_test)
y_pred_rf = rf.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

print("Confusion Matrix (RF):")
print(confusion_matrix(y_test, y_pred_rf))

print("Classification Report:")
print(classification_report(y_test, y_pred_rf))